In [1]:
import torch
import torch.nn as nn
import math
from torchvision import models, transforms
from torch.utils.data import DataLoader, Subset, Dataset, WeightedRandomSampler
import os
import numpy as np
import pandas as pd
import nibabel as nib
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from util import filter_data, seed_everything, report_split_stats
from util import mask_crop as mask_crop_fn
from validate import val_model_stable as val_model

# ============================================================
# EXPERIMENT CONFIG
# ============================================================
EXP_NAME = "true_conditional_diffusion_ae" 
device = 'cuda:0'
LIMIT_SUBS = None 
CACHE_PATH = 'qsm_preprocessed_cache.pt'
LOAD_FROM_CACHE = True 
seed_everything(42)

# ============================================================
# NEW: SINUSOIDAL TIME EMBEDDING
# ============================================================
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = t[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

# ============================================================
# 1. DATASET CLASS (Standard QSM)
# ============================================================
class QSM_RAM_Dataset(Dataset):
    def __init__(self, nii_dir, seg_dir, mask_crop_fn, clinical_dict, label_map, limit=None, cache_path=None, load_cache=False):
        self.samples = []
        self.volumes = {} 
        self.clinical_dict = clinical_dict
        self.label_map = label_map
        self.transform = None   
        self.train_mode = False 
        
        if load_cache and cache_path and os.path.exists(cache_path):
            print(f">>> Loading preprocessed data from cache: {cache_path}...")
            cached_data = torch.load(cache_path)
            self.volumes = cached_data['volumes']
            self.samples = cached_data['samples']
            return 

        all_potential = [f for f in os.listdir(nii_dir) if f.startswith('qsm_') and f.endswith('.nii.gz')]
        for f in tqdm(all_potential, desc="Caching Volumes"):
            try:
                sub_id = int(f.split('_')[1])
                case_id = f"{sub_id:02d}"
                mask_path = os.path.join(seg_dir, f'seg_{case_id}.nii.gz')
                if not os.path.exists(mask_path): continue
                raw_data = nib.load(os.path.join(nii_dir, f)).get_fdata()
                mask_data = nib.load(mask_path).get_fdata()
                mask_data[mask_data <= 2] = 0
                img = mask_crop_fn(raw_data, mask_data, (72, 64, 64)) / 1000.0
                binary_mask = (mask_crop_fn((mask_data > 0).astype(np.uint8), mask_data, (72, 64, 64)) > 0)
                if img.shape != (72, 64, 64): continue
                if np.any(binary_mask):
                    img = (img - np.mean(img[binary_mask])) / (np.std(img[binary_mask]) + 1e-8)
                self.volumes[sub_id] = np.transpose(np.clip(img, -5.0, 5.0), (1, 2, 0)).astype(np.float32)
                actual_label = self.label_map.get(sub_id, -1)
                for slice_idx in range(72):
                    self.samples.append({'sub_id': sub_id, 'slice_idx': slice_idx, 'label': actual_label})
            except Exception: continue
        if cache_path: torch.save({'volumes': self.volumes, 'samples': self.samples}, cache_path)

    def __len__(self): return len(self.samples)
    def __getitem__(self, index):
        meta = self.samples[index]
        img = self.volumes[meta['sub_id']][:, :, meta['slice_idx']]
        img_tensor = torch.from_numpy(img).unsqueeze(0) 
        if self.train_mode and self.transform: img_tensor = self.transform(img_tensor)
        img_tensor = img_tensor / 5.0 
        clin_data = self.clinical_dict.get(str(meta['sub_id']))
        clin_vec = torch.tensor(clin_data, dtype=torch.float32) if clin_data is not None else torch.zeros(getattr(self, 'clin_dim', 11))
        return img_tensor, clin_vec, int(meta['label']), index

# ============================================================
# 2. MODELS (CONDITIONAL DIFFUSION)
# ============================================================
class QSMDecoder(nn.Module):
    def __init__(self, feat_dim=512):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(feat_dim, 256, 4, 1, 0), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.decoder(x)

class ConditionalResNet(nn.Module):
    def __init__(self, model, clinical_dim, time_dim=128):
        super().__init__()
        self.base_model = model
        self.feat_dim = model.fc.in_features
        self.base_model.fc = nn.Identity()
        
        self.time_mlp = nn.Sequential(TimeEmbedding(time_dim), nn.Linear(time_dim, time_dim), nn.ReLU())
        
        # Condition fusion: Clinical + Time + Image Features
        self.condition_projector = nn.Sequential(
            nn.Linear(self.feat_dim + clinical_dim + time_dim, self.feat_dim),
            nn.ReLU()
        )
        
        self.classifier_head = nn.Sequential(
            nn.Linear(self.feat_dim, 256), nn.ReLU(),
            nn.Dropout(0.4), nn.Linear(256, 2)
        )

    def forward(self, x, clinical_vec, t):
        visual_feats = self.base_model(x)
        t_emb = self.time_mlp(t)
        
        # Concatenate all three sources of information
        combined = torch.cat([visual_feats, clinical_vec, t_emb], dim=1)
        fused_feats = self.condition_projector(combined)
        
        logits = self.classifier_head(fused_feats)
        return logits, fused_feats.view(fused_feats.size(0), self.feat_dim, 1, 1)

# ============================================================
# 3. RUNTIME & DIFFUSION SETUP
# ============================================================
# ============================================================
# 3. RUNTIME & CV
# ============================================================
nii_path = '/data2/ali/dbs/qsm/'
seg_path = '/data2/ali/dbs/seg_ps/'
file_dir = '/data2/ali/dbs/dbs_03292024.csv'

cv_features = {'Age', 'Sex', 'Ethnicity', 'Race', 'Disease Duration (year)', 'Physician', ' pre op levadopa equivalent dose (mg)', ' Location', ' Target', ' Test medication status', ' ON (pre-dbs updrs)'}
all_needed_cols = cv_features | {'CORNELL ID', ' OFF meds ON stim 6mo'}

motor_df = filter_data(file_dir, all_needed_cols, True)
motor_df[' ON (pre-dbs updrs)'] = pd.to_numeric(motor_df[' ON (pre-dbs updrs)'], errors='coerce')
motor_df[' OFF meds ON stim 6mo'] = pd.to_numeric(motor_df[' OFF meds ON stim 6mo'], errors='coerce')
motor_df = motor_df.dropna(subset=[' ON (pre-dbs updrs)', ' OFF meds ON stim 6mo'])

improvement_ratios = (motor_df[' ON (pre-dbs updrs)'] - motor_df[' OFF meds ON stim 6mo']) / motor_df[' ON (pre-dbs updrs)']
label_map = {int(row['CORNELL ID']): (1 if ratio >= 0.30 else 0) for (_, row), ratio in zip(motor_df.iterrows(), improvement_ratios)}
# --- CLINICAL DATA CLEANING & NORMALIZATION ---
cols_to_norm = ['Age', 'Disease Duration (year)', ' ON (pre-dbs updrs)', ' pre op levadopa equivalent dose (mg)']

for col in cols_to_norm:
    # 1. Force to numeric (converts strings to floats, non-numeric to NaN)
    motor_df[col] = pd.to_numeric(motor_df[col], errors='coerce')
    
    # 2. Handle missing values (important for Z-scoring)
    col_mean = motor_df[col].mean()
    motor_df[col] = motor_df[col].fillna(col_mean)
    
    # 3. Z-score Normalize: (Value - Mean) / Std
    col_std = motor_df[col].std()
    motor_df[col] = (motor_df[col] - col_mean) / (col_std + 1e-8)

print(">>> Clinical features normalized successfully.")
clinical_dict = {str(int(row['CORNELL ID'])): row[list(cv_features)].values.astype(np.float32) 
                 for _, row in motor_df.iterrows()}
full_dataset = QSM_RAM_Dataset(nii_path, seg_path, mask_crop_fn, clinical_dict, label_map, limit=LIMIT_SUBS, cache_path=CACHE_PATH, load_cache=LOAD_FROM_CACHE)
actual_clin_dim = next(iter(clinical_dict.values())).shape[0]
full_dataset.clin_dim = actual_clin_dim

all_cached_ids = set(full_dataset.volumes.keys())
labeled_subs = np.array(list(all_cached_ids & set(label_map.keys())))
unlabeled_ids = np.array(list(all_cached_ids - set(label_map.keys())))
sub_labels = np.array([label_map[sid] for sid in labeled_subs])

qsm_aug = transforms.Compose([
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05))
])

class DiffusionHelper:
    def __init__(self, timesteps=100, device='cuda'):
        self.timesteps = timesteps
        self.device = device
        self.beta = torch.linspace(0.0001, 0.02, timesteps).to(device)
        self.alpha_cumprod = torch.cumprod(1.0 - self.beta, dim=0)

    def add_noise(self, x_start, t):
        noise = torch.randn_like(x_start)
        sqrt_alpha_cumprod_t = torch.sqrt(self.alpha_cumprod[t]).view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_cumprod_t = torch.sqrt(1 - self.alpha_cumprod[t]).view(-1, 1, 1, 1)
        return sqrt_alpha_cumprod_t * x_start + sqrt_one_minus_alpha_cumprod_t * noise, noise

diff_helper = DiffusionHelper(timesteps=100, device=device)

# ============================================================
# 4. TRAINING LOOP
# ============================================================
all_split_best_metrics = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for split, (t_p_idx, v_p_idx) in enumerate(skf.split(labeled_subs, sub_labels)):
    train_subs, val_subs = labeled_subs[t_p_idx], labeled_subs[v_p_idx]
    report_split_stats(train_subs, val_subs, motor_df)
    pt_subs = np.concatenate([unlabeled_ids, train_subs])
    
    t_idx = [i for i, s in enumerate(full_dataset.samples) if s['sub_id'] in train_subs]
    v_idx = [i for i, s in enumerate(full_dataset.samples) if s['sub_id'] in val_subs]
    pt_idx = [i for i, s in enumerate(full_dataset.samples) if s['sub_id'] in pt_subs]

    pt_loader = DataLoader(Subset(full_dataset, pt_idx), batch_size=48, shuffle=True)
    train_labels = [label_map[full_dataset.samples[i]['sub_id']] for i in t_idx]
    class_weights = 1. / torch.tensor(np.bincount(train_labels), dtype=torch.float)
    sampler = WeightedRandomSampler([class_weights[l] for l in train_labels], 2*len(t_idx))
    t_loader = DataLoader(Subset(full_dataset, t_idx), batch_size=48, sampler=sampler)
    v_loader = DataLoader(Subset(full_dataset, v_idx), batch_size=48, shuffle=False)

    print(f"\n>>> Split {split} | PT Subs: {len(pt_subs)} | Val Subs: {len(val_subs)}")
    # --- A. CONDITIONAL DIFFUSION PRETRAINING ---
    base_resnet = models.resnet18(pretrained=True)
    base_resnet.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
    model = ConditionalResNet(base_resnet, clinical_dim=actual_clin_dim).to(device)
    decoder = QSMDecoder(feat_dim=512).to(device)
    
    optimizer_pt = torch.optim.Adam(list(model.parameters()) + list(decoder.parameters()), lr=1e-4)
    criterion_pt = nn.MSELoss()

    print(">>> Starting Conditional Diffusion Denoising Pretraining...")
    for pt_epoch in range(10):
        model.train(); decoder.train()
        for imgs, clin, _, _ in pt_loader:
            imgs, clin = imgs.to(device), clin.to(device)
            t = torch.randint(0, diff_helper.timesteps, (imgs.size(0),), device=device).long()
            
            noisy_imgs, _ = diff_helper.add_noise(imgs, t)
            optimizer_pt.zero_grad()
            
            # Forward: Use Noisy Image + Clinical Vector + Time Step
            _, fused_latent = model(noisy_imgs, clin, t)
            recon = decoder(fused_latent)
            
            loss = criterion_pt(recon, imgs)
            loss.backward(); optimizer_pt.step()

    # --- B. FINE-TUNING (Conditioned Classifier) ---
    # Freeze ResNet and Projector, train only Classifier Head
    for param in model.base_model.parameters(): param.requires_grad = False
    for param in model.condition_projector.parameters(): param.requires_grad = False

    optimizer = torch.optim.Adam(model.classifier_head.parameters(), lr=5e-5)
    loss_fn = nn.CrossEntropyLoss().to(device)

    best_f1, patience, best_metrics_this_split = 0, 0, None
    os.makedirs(f"weights/{EXP_NAME}", exist_ok=True)

    for epoch in range(30):
        model.train(); full_dataset.train_mode = True
        for imgs, clin, lbls, _ in t_loader:
            imgs, clin, lbls = imgs.to(device), clin.to(device), lbls.to(device)
            optimizer.zero_grad()
            logits, _ = model(imgs, clin)
            loss_fn(logits, lbls).backward(); optimizer.step()
        
        model.eval(); full_dataset.train_mode = False
        wrapped_v_loader = ValLoaderWrapper(v_loader)
        m = val_model(wrapped_v_loader, device, model, loss_fn, v_loader.dataset, threshold=0.5)
        current_f1 = 2*(m[2]*m[3])/(m[2]+m[3]) if (m[2]+m[3])>0 else 0

        if current_f1 > best_f1:
            best_f1, best_metrics_this_split, patience = current_f1, m, 0
            torch.save(model.state_dict(), f"weights/{EXP_NAME}/best_f1_split_{split}.pth")
        else: patience += 1

        print(f"Split {split} Ep {epoch} | F1: {current_f1:.4f} | AUC: {m[5]:.4f} | Acc: {m[1]:.4f}")
        if patience >= 10: break
    
    if best_metrics_this_split is not None: all_split_best_metrics.append(best_metrics_this_split)

# ============================================================
# FINAL SUMMARY
# ============================================================
final_metrics = np.array(all_split_best_metrics)
avg_metrics, std_metrics = np.mean(final_metrics, axis=0), np.std(final_metrics, axis=0)
print("\n" + "="*45 + "\nFINAL CV SUMMARY (BEST F1 PER SPLIT)\n" + "="*45)
names = ["Loss", "Accuracy", "Precision", "Sensitivity", "Specificity", "AUC"]
for i, name in enumerate(names):
    print(f"{name:<15} : {avg_metrics[i]:.4f} ± {std_metrics[i]:.4f}")
print("="*45)

IndentationError: unexpected indent (1406874911.py, line 168)